# Task 2 — Explaining household electricity consumption predictions with SHAP

Use SHAP to explain the Main Model overall feature contributions and selected hourly predictions.

**Prediction target:** hourly household electricity consumption, in kWh. 
**Features:** outdoor temperature, hour of day, day of week, and consumption 24 hours earlier. 
**Model:** standardized multiple linear regression, reproduced without changing the Task 1 specification.

The notebook is standalone, and it uses the same datasets and preprocessing choices as Task 1 (model.ipynb).

## 1. Setup and data requirements

Install packages if needed: `pip install numpy pandas matplotlib scikit-learn shap ucimlrepo`.

Place `sceaux_hourly_temperature.csv` in `data/` relative to the notebook. Household power readings are fetched from the UCI repository (dataset ID 235), so the first run requires internet access.

**Assumptions:** 
- Household readings have no explicitly specified timezone; matching them to Open-Meteo timestamps assumes the same clock basis. 
- Temperature is observed at the prediction hour, not a forecast. 

In [10]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from IPython.display import display
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

DATA = Path("data")
assert (DATA / "sceaux_hourly_temperature.csv").is_file(), (
    "Place sceaux_hourly_temperature.csv in data/ relative to the notebook."
)
TARGET = "Global_active_power"
FEATURES_MAIN = ["temperature", "hour", "day_of_week", "consumption_lag_24h"]

def scores(y, prediction):
    return {
        "MAE [kWh]": mean_absolute_error(y, prediction),
        "RMSE [kWh]": root_mean_squared_error(y, prediction),
    }

print("SHAP version:", shap.__version__)

/Users/angelina/UHK/Masters Study/UHK2_Sem3/SU/ukol1/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SHAP version: 0.54.1


## 2. Data loading and feature preparation

Preserved from Task 1: 
- parse minute-level active power (kW)
- aggregate to hourly estimated energy (kWh) only when at least 55 valid minute readings are present
- create the 24-hour lag over a continuous hourly timeline
- merge temperature
- add calendar features. 

Dropping incomplete cases happens after lag creation, so a lag means 24 hours earlier.

With one-minute kW readings, the hourly mean kW is numerically equal to the estimated kWh consumed over a one-hour interval.

In [4]:
# Source loading — equivalent to Task 1, without intermediate audits.
df_consumption = fetch_ucirepo(id=235).data.original.replace(["", "?"], pd.NA)
df_temperature = pd.read_csv(DATA / "sceaux_hourly_temperature.csv")

minute_data = df_consumption[["Date", "Time", TARGET]].copy()
minute_data[TARGET] = pd.to_numeric(minute_data[TARGET], errors="coerce")
minute_data["time"] = pd.to_datetime(
    minute_data["Date"] + " " + minute_data["Time"],
    format="%d/%m/%Y %H:%M:%S",
).dt.floor("h")

# resample creates all intervening hours, including hours without valid readings
hourly = minute_data.set_index("time")[TARGET].resample("h")
valid_measurements = hourly.count()
t = 5 # threshold
total_measurements = 60
consumption_hourly = (
    hourly.mean()
    .where(valid_measurements >= (total_measurements - t))  # same as in task 1
    .rename(TARGET)
    .reset_index()
)
consumption_hourly["consumption_lag_24h"] = consumption_hourly[TARGET].shift(24)

weather = df_temperature.rename(columns={"temperature_2m (°C)": "temperature"}).copy()
weather["time"] = pd.to_datetime(weather["time"])
assert weather["time"].is_unique, "Temperature timestamps must be unique."

df_master = consumption_hourly.merge(
    weather[["time", "temperature"]], on="time", how="left", validate="one_to_one"
)
df_master["hour"] = df_master["time"].dt.hour
df_master["day_of_week"] = df_master["time"].dt.dayofweek  # Monday=0

df_model = df_master.dropna(subset=[TARGET] + FEATURES_MAIN).copy()
assert df_master["time"].is_unique
assert np.isfinite(df_model[FEATURES_MAIN + [TARGET]].to_numpy(dtype=float)).all()
print("Complete usable hours:", len(df_model), "of", len(df_master))
display(df_model[["time"] + FEATURES_MAIN + [TARGET]].head())

/Users/angelina/UHK/Masters Study/UHK2_Sem3/SU/ukol1/.venv/lib/python3.14/site-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (2: Global_active_power, 3: Global_reactive_power, 4: Voltage, 5: Global_intensity, 6: Sub_metering_1, 7: Sub_metering_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


Complete usable hours: 33945 of 34589


,time,temperature,hour,day_of_week,consumption_lag_24h,Global_active_power
25,2006-12-17 18:00:00,4.7,18,6,3.632200,3.697100
26,2006-12-17 19:00:00,4.7,19,6,3.400233,2.908400
27,2006-12-17 20:00:00,4.2,20,6,3.268567,3.361500
28,2006-12-17 21:00:00,3.7,21,6,3.056467,3.040767
29,2006-12-17 22:00:00,3.5,22,6,2.200133,1.518000


## 3. Chronological split and model reproduction

The first 70% of the full hourly timeline is in training, the next 15% is validation, and the final 15% is test. Rows with missing values are removed within these time periods. 

Fit the scaler and regression only on training data, the fitted scaler is then applied to validation and test.

In [6]:
timeline = df_master["time"].sort_values().reset_index(drop=True)
train_end = timeline.iloc[int(len(timeline) * 0.70)]
val_end = timeline.iloc[int(len(timeline) * 0.85)]

train = df_model.loc[df_model["time"] < train_end].sort_values("time").copy()
val = df_model.loc[(df_model["time"] >= train_end) & (df_model["time"] < val_end)].sort_values("time").copy()
test = df_model.loc[df_model["time"] >= val_end].sort_values("time").copy()

assert train["time"].max() < val["time"].min() < test["time"].min()
assert all(part[FEATURES_MAIN + [TARGET]].notna().all().all() for part in (train, val, test))
print("Train / validation / test hours:", len(train), len(val), len(test))

Train / validation / test hours: 23962 5052 4931


Scaler fit on train and model fit. 

In [7]:
scaler_main = StandardScaler().fit(train[FEATURES_MAIN])

def scaled(frame):
    return pd.DataFrame(
        scaler_main.transform(frame[FEATURES_MAIN]),
        index=frame.index, columns=FEATURES_MAIN,
    )

z_train_main, z_val_main, z_test_main = map(scaled, (train, val, test))
y_train, y_val, y_test = (part[TARGET] for part in (train, val, test))

model_main = LinearRegression().fit(z_train_main, y_train)
pred_train_main = model_main.predict(z_train_main)
pred_val_main = model_main.predict(z_val_main)
pred_test_main = model_main.predict(z_test_main)

performance = pd.DataFrame(
    [scores(y_train, pred_train_main), scores(y_val, pred_val_main), scores(y_test, pred_test_main)],
    index=pd.Index(["Train", "Validation", "Test"], name="Split"),
)
display(performance.round(3))

,MAE [kWh],RMSE [kWh]
Split,,
Train,0.605,0.796
Validation,0.603,0.778
Test,0.491,0.636


### Model coefficients (reference for SHAP)

The regression coefficients below represent the prediction change in **kWh per one training-set standard deviation of a feature** (with all other standardized inputs fixed). 

Coefficients show model behavior, not causal effects.

In [8]:
coefficients = pd.DataFrame({
    "Feature": FEATURES_MAIN,
    "Coefficient [kWh / 1 SD]": model_main.coef_,
    "Training mean (original units)": scaler_main.mean_,
    "Training SD (original units)": scaler_main.scale_,
}).set_index("Feature")
display(coefficients.round(4))
print("Intercept [kWh]:", round(float(model_main.intercept_), 4))

,Coefficient [kWh / 1 SD],Training mean (original units),Training SD (original units)
Feature,,,
temperature,-0.1738,11.3687,6.8509
hour,0.2398,11.5010,6.9275
day_of_week,0.0797,2.9965,1.9952
consumption_lag_24h,0.2888,1.0864,0.9285


Intercept [kWh]: 1.0863
